<a href="https://colab.research.google.com/github/ekrasafdar/AIMentor/blob/main/brain_tumor_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
zip_path = '/content/drive/MyDrive/brain_tumor_research/archive.zip'
extract_path = '/content/brain_tumor_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted!")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    if level <= 2:
        print('  '*level + os.path.basename(root) + '/', f'({len(files)} files)' if files else '')

Mounted at /content/drive
✅ Extracted!
brain_tumor_data/ 
  Testing/ 
    notumor/ (400 files)
    pituitary/ (400 files)
    meningioma/ (400 files)
    glioma/ (400 files)
  Training/ 
    notumor/ (1400 files)
    pituitary/ (1400 files)
    meningioma/ (1400 files)
    glioma/ (1400 files)


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print("✅ Ready")

✅ Ready


In [3]:
train_path = '/content/brain_tumor_data/Training'
test_path = '/content/brain_tumor_data/Testing'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, validation_split=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)
val_generator = train_datagen.flow_from_directory(
    train_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)
test_generator = test_datagen.flow_from_directory(
    test_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
print(f"✅ Train: {train_generator.samples}, Val: {val_generator.samples}, Test: {test_generator.samples}")
print(f"Classes: {train_generator.class_indices}")

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
✅ Train: 4480, Val: 1120, Test: 1600
Classes: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [ ]:
def build_model(base_model_class, num_classes):
    base = base_model_class(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(base.input, out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

results = {}
trained_models = {}

for name, base_cls in [('ResNet50', ResNet50), ('EfficientNetB0', EfficientNetB0), ('MobileNetV2', MobileNetV2)]:
    print(f"\n🟢 Training {name}...")
    model = build_model(base_cls, 4)
    model.fit(train_generator, validation_data=val_generator, epochs=10, verbose=1)

    test_generator.reset()
    pred_probs = model.predict(test_generator, verbose=0)
    pred = np.argmax(pred_probs, axis=1)
    true = test_generator.classes

    p, r, f, _ = precision_recall_fscore_support(true, pred, average='weighted')
    acc = np.mean(pred == true)
    results[name] = {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f}
    trained_models[name] = model
    print(f"✅ {name} — Acc: {acc:.4f}, F1: {f:.4f}")

import pandas as pd
print("\n🏆 FINAL RESULTS:")
print(pd.DataFrame(results).T)


🟢 Training ResNet50...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 103s 615ms/step - accuracy: 0.4114 - loss: 1.2336 - val_accuracy: 0.4705 - val_loss: 1.1120
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 80s 568ms/step - accuracy: 0.4955 - loss: 1.1055 - val_accuracy: 0.4938 - val_loss: 1.0377
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 560ms/step - accuracy: 0.5482 - loss: 1.0367 - val_accuracy: 0.6018 - val_loss: 0.9676
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 81s 573ms/step - accuracy: 0.5670 - loss: 1.0099 - val_accuracy: 0.5625 - val_loss: 0.9958
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 557ms/step - accuracy: 0.5676 - loss: 0.9958 - val_accuracy: 0.5750 - val_loss: 0.9610
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 80s 573ms/step - accuracy: 0.5967 - loss: 0.9561 - val_accuracy: 0.5571 - val_loss: 0.9520
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 554ms/step - accuracy: 0.5998 - loss: 0.9432 - val_accuracy: 0.5750 - val_loss: 1.0050
Epoch 8